# dev... Quantum Krylov subspace method

ライブラリのimportとPairingHamiltonianのクラスを定義しておく↓

In [1]:
import numpy as np
import scipy
import itertools
from itertools import combinations
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import PauliEvolutionGate, PauliGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.visualization import plot_histogram
import seaborn as sns
from tqdm import tqdm
from qiskit_aer.primitives import SamplerV2

class PairingHamiltonian:
    def __init__(self, Norb, Nocc, gval, delta_eps=1.0):
        self.Norb = Norb
        self.Nocc = Nocc
        self.delta_eps = delta_eps
        self.gval = gval
        self.basis = self.make_basis()
        self.epsilon = self.eval_epsilon()
        self.Hmat = self.eval_Hmat()

    def make_basis(self):
        self.basis = []
        for occ in combinations(range(self.Norb), self.Nocc):
            self.basis.append(occ)

        return self.basis
    
    def eval_epsilon(self):
        self.epsilon = [ 2 * i * self.delta_eps for i in range(self.Norb) ]
        return self.epsilon
    
    def eval_Hmat(self):
        dim = len(self.basis)
        self.Hmat = np.zeros((dim, dim))
        for bra_idx, bra in enumerate(self.basis):
            for ket_idx, ket in enumerate(self.basis):
                # Hamming distance
                diff = [ i for i in bra if i not in ket ]
                same = [ i for i in bra if i in ket ]
                # for SPE term
                if bra_idx == ket_idx:
                    self.Hmat[bra_idx, ket_idx] += np.sum( [self.epsilon[i] for i in same])
                    self.Hmat[bra_idx, ket_idx] += - self.gval * len(same) 
                # for pairing term
                if len(diff) == 1:
                    self.Hmat[bra_idx, ket_idx] = - self.gval

        return self.Hmat

def tuple_to_bitstring(tup, Norb, rev=True):
    bitint = 0
    for i in tup:
        bitint += 2**i
    if rev:
        bitstring = "|"+format(bitint, f'0{Norb}b')[::-1]+">"
    else:
        bitstring = "|"+format(bitint, f'0{Norb}b')+">"        
    return bitstring

def cG1(circ, c_qubit, i, j, theta):
    theta_4 = theta / 4 
    circ.cx(i,j)
    circ.ry(theta_4, i)
    circ.cx(j,i)
    circ.ry(-theta_4, i)
    circ.cx(c_qubit, i)
    circ.ry(theta_4, i)
    circ.cx(j,i)
    circ.ry(-theta_4, i)
    circ.cx(c_qubit, i)
    circ.cx(i,j)

def cG1_gate(theta):
    circ = QuantumCircuit(2)
    G(circ, 0, 1, theta)
    circ.name = "cG1"   
    circ = circ.to_gate()
    circ = circ.control(1) 
    return circ

def G(circ, i, j, theta):
    theta_2 = theta / 2 
    circ.cx(i,j)
    circ.ry(theta_2, i)
    circ.cx(j,i)
    circ.ry(-theta_2, i)
    circ.cx(j,i)
    circ.cx(i,j)  

def G_gate(theta):
    circ = QuantumCircuit(2)
    G(circ, 0, 1, theta)
    circ.name = "G"    
    return circ.to_gate()

def get_idx_ancilla_in_string(n_qubit, ancilla, Qiskit_ordering):
    idx_ancilla = None
    if ancilla != None:
        if Qiskit_ordering:
            idx_ancilla = n_qubit-1 - ancilla
        else:
            idx_ancilla = ancilla
    return idx_ancilla

def expec_Zstring(res, idx_relevant, Qiskit_ordering=True, target_qubits=[], ancilla_qubit=None):
    exp_val = exp_val_p0 = exp_val_p1 = 0.0
    n_shot = sum(res.values())
    n_qubit = len(list(res.keys())[0])
    idx_ancilla = get_idx_ancilla_in_string(n_qubit, ancilla_qubit, Qiskit_ordering)
    for bitstr, count in res.items():
        if ancilla_qubit != None and target_qubits != []:
            bitstr_target = ''.join([bitstr[k] for k in range(n_qubit) if k != idx_ancilla])
        else:
            bitstr_target = bitstr
        tmp = 1.0
        for idx in idx_relevant:
            if Qiskit_ordering:
                idx = -1 - idx
            bit = int(bitstr_target[idx])            
            tmp *= (1 - 2 * bit)
        exp_val += tmp * count
        
        if ancilla_qubit != None:
            if int(bitstr[idx_ancilla]) == 0:
                exp_val_p0 += tmp * count
            else:
                exp_val_p1 += tmp * count
    exp_val /= n_shot; exp_val_p0 /= n_shot; exp_val_p1 /= n_shot
    return exp_val, exp_val_p0, exp_val_p1


params_exact = np.array([-0.48104276, -1.03976498, -0.98963981, -1.18481738, -0.54832984])

Norb = 4
Nocc = 2
gval = 0.33  

Hamil = PairingHamiltonian(Norb, Nocc, gval)
evals, evecs = np.linalg.eigh(Hamil.Hmat)
evals = np.linalg.eigvalsh(Hamil.Hmat)
Egs_exact = evals[0]
E_HF = Hamil.Hmat[0,0]

SPEs = Hamil.epsilon
pauli_list = [ ]
obs = [ ]
coeffs = [ ]

# I term
coeff = 0.0
op = "I" * Norb
for i in range(Norb):
    coeff += 0.5 * ( SPEs[i] - Hamil.gval ) 
obs += [op]
coeffs += [coeff]
# -Zp term
for i in range(Norb):
    op = "I" * Norb
    op = op[:i] + "Z" + op[i+1:]
    coeff = -0.5 * ( SPEs[i] - Hamil.gval )

    op = op[::-1]
    obs += [op]
    coeffs += [coeff]
# XX+YY term
for i in range(Hamil.Norb):
    for j in range(i+1, Hamil.Norb):
        factor = - Hamil.gval / 2
        op = "I" * Norb
        op = op[:i] + "X" + op[i+1:j] + "X" + op[j+1:]
        op = op[::-1]
        obs += [op]
        coeffs += [ factor ]
        op = "I" * Norb
        op = op[::-1]
        op = op[:i] + "Y" + op[i+1:j] + "Y" + op[j+1:]
        obs += [op]
        coeffs += [ factor ]

hamiltonian_op = SparsePauliOp(obs, coeffs)

# print("basis:", Hamil.basis)
# print([tuple_to_bitstring(tup, Norb) for tup in Hamil.basis])
# print("eps: ", Hamil.epsilon)
# print("Hmat: ", Hamil.Hmat)
print("evals: ", evals)
# print("Egs_exact: ", Egs_exact, " E_HF", E_HF)
# print("gs evec", evecs[:,0])
# print("gs prob", evecs[:,0]**2)


evals:  [1.18985184 3.29649666 5.34       5.34       7.42853393 9.44511758]


In [2]:
def additional_qc(qc_in, pauli_str, register_target, Qiskit_order=True):
    pauli_str = str(pauli_str)
    if Qiskit_order:
        pauli_str = pauli_str[::-1]

    for i in range(len(pauli_str)):
        if pauli_str[i] == "X":
            qc_in.h(register_target[i])
        elif pauli_str[i] == "Y":
            qc_in.sdg(register_target[i]); qc_in.h(register_target[i])
        elif pauli_str[i] == "Z" or pauli_str[i] == "I":
            pass
        else:
            raise ValueError("Invalid Pauli string: ", pauli_str)

def get_idx_to_measure(pauli_str, Qiskit_order=True):
    idxs = [ idx for idx, p in enumerate(pauli_str) if p != "I"]
    if Qiskit_order:
        idxs = [ len(pauli_str) - 1 - idx for idx in idxs]
    return idxs



In [3]:
def ansatz(params, method="FCI"):
    qc = QuantumCircuit(Norb)
    # HF
    for i in range(Nocc):
        qc.x(i)
    # Additional gates
    if method == "FCI":
        if Norb != 4 or Nocc != 2:
            print("You are using parameters to reproduce the FCI wavefunction for Norb=4 and Nocc=2")
            print("Please make sure whether this is correct for your case!")
        qc.append(G_gate(params[0]), [1, 2])
        qc.append(G_gate(params[1]), [2, 3])
        qc.append(cG1_gate(params[2]), [2, 0, 1])
        qc.append(cG1_gate(params[3]), [3, 0, 1])
        qc.append(cG1_gate(params[4]), [3, 1, 2])        
    return qc

Toeplitz構造を課す方法:

N. Yoshioka et al.: "Krylov diagonalization of large many-body Hamiltonians on a quantum processor", Nature Communications 16, 5014 (2025) 

$$
\langle \psi_0 | U^\dagger_j H U_k | \psi_0 \rangle = \langle \psi_0 | H U_{k-j} | \psi_0 \rangle 
$$

を用いる。形式的には正しいが、実用的には $U$ がTrotter分解を用いて近似的な実時間発展になるので、simulatorや実機で良い結果を与える保証はない。
ただし、計算量的にはぐっと楽になる。

In [4]:
# def make_U_and_cU(i, delta_t, trotter_order, trotter_steps, hamiltonian_op, ancilla_qubits, target_qubits, Uprep):
#     Ntar = len(target_qubits)
#     time = i * delta_t
#     circuit_U = QuantumCircuit(Ntar)
#     expiHt = PauliEvolutionGate(hamiltonian_op, time, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
#     circuit_U.append(expiHt, range(Ntar))
#     qc_U = circuit_U.decompose().decompose()
    
#     qc_cU = QuantumCircuit(Ntar)
#     qc_cU.append(Uprep, range(Ntar))
#     qc_cU.append(expiHt, range(Ntar))
#     qc_cU = qc_cU.decompose().decompose()
#     qc_cU = qc_cU.to_gate().control(1)

#     return qc_U, qc_cU

def make_overlap_qc_conventional(Ntar, gate_cUi, gate_cUj, ancilla_qubits, target_qubits, using_statevector):
    qc_re = QuantumCircuit(1+Ntar, 1)
    qc_re.h(0)
    qc_re.append(gate_cUi, ancilla_qubits + target_qubits)
    qc_re.x(0)
    qc_re.append(gate_cUj, ancilla_qubits+target_qubits)
    qc_re.h(0)
    if not using_statevector:
        qc_re.measure(0,0)
    qc_re = qc_re.decompose()

    # Overlap, Im part 
    qc_im = QuantumCircuit(1+Ntar,1)
    qc_im.h(0)
    qc_im.append(gate_cUi, ancilla_qubits + target_qubits)
    qc_im.x(0)
    qc_im.append(gate_cUj, ancilla_qubits + target_qubits)
    qc_im.sdg(0)
    qc_im.h(0)
    if not using_statevector:
        qc_im.measure(0,0)
    qc_im = qc_im.decompose()

    return qc_re, qc_im



def make_overlap_qc(Ntar, gate_cPrep, gate_Ukj, Uj, ancilla_qubits, target_qubits, using_statevector):
    qc_re = QuantumCircuit(1+Ntar, 1)
    qc_re.h(0)
    qc_re.append(gate_cPrep, ancilla_qubits + target_qubits)
    qc_re.append(gate_Ukj, target_qubits)
    qc_re.x(0)
    qc_re.append(gate_cPrep, ancilla_qubits + target_qubits)
    qc_re.x(0)
    qc_re.h(0)
    if not using_statevector:
        qc_re.measure(0,0)
    qc_re = qc_re.decompose()

    # Overlap, Im part 
    qc_im = QuantumCircuit(1+Ntar,1)
    qc_im.h(0)
    qc_im.append(gate_cPrep, ancilla_qubits + target_qubits)
    qc_im.append(gate_Ukj, target_qubits)
    qc_im.x(0)
    qc_im.append(gate_cPrep, ancilla_qubits + target_qubits)
    qc_im.x(0)
    qc_im.sdg(0)
    qc_im.h(0)
    if not using_statevector:
        qc_im.measure(0,0)
    qc_im = qc_im.decompose()
    return qc_re, qc_im

def measure_overlap(num_shot, Ntar, gate_cUi, gate_cUj, ancilla_qubits, target_qubits, sampler, using_statevector,
                    do_simulation=True):
    qc_re, qc_im = make_overlap_qc(Ntar, gate_cUi, gate_cUj, ancilla_qubits, target_qubits, using_statevector)
    if do_simulation:
        if using_statevector:
            results = [ Statevector.from_instruction(qc).probabilities_dict() for qc in [qc_re, qc_im]]
            prob_Re = results[0]
            prob_Im = results[1] 
        else:
            print("sampler used 1")
            job = sampler.run([qc_re, qc_im], shots=num_shot)
            results  = job.result()
            prob_Re = results[0].data.c.get_counts()
            prob_Im = results[1].data.c.get_counts()

        p0 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Re.values()))
        p1 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Re.values()))
        ReN = p0 - p1

        p0 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Im.values()))
        p1 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Im.values()))
        ImN = p0 - p1

        U_ij = ReN + 1j * ImN
        return U_ij
    else: #only resource estimation
        print("qc_re:", dict(qc_re.count_ops()))
        print("qc_im:", dict(qc_im.count_ops()))
        return None

def make_cU(Uprep, Ui, Ntar, only_Uprep=False):
    circuit_cUi = QuantumCircuit(Ntar)
    circuit_cUi.append(Uprep, range(Ntar))
    if not only_Uprep:
        circuit_cUi.append(Ui, range(Ntar))
    circuit_cUi = circuit_cUi.decompose(reps=3)
    return circuit_cUi.to_gate().control(1)

def make_Circ_forNondiagH(term_types, Ntar, ancilla_qubits, target_qubits, 
                          cPrep, U_kj, U_j, qcs_re, qcs_im, using_statevector):
    
    for idx_term in range(len(term_types)):
        term = term_types[idx_term]

        qc = QuantumCircuit(1+Ntar)
        qc.h(0)
        qc.append(cPrep, range(Ntar+1))
        qc.append(U_kj,  target_qubits)
        qc.x(0)
        qc.append(cPrep, range(Ntar+1))
        qc.x(0)
        qc.h(0)

        qc_im = QuantumCircuit(1+Ntar)
        qc_im.h(0)
        qc_im.append(cPrep, range(Ntar+1))
        qc_im.append(U_kj,  target_qubits)
        qc_im.x(0)
        qc_im.append(cPrep, range(Ntar+1))
        qc_im.x(0)
        qc_im.sdg(0)
        qc_im.h(0)

        if term == "IZ":
            pass
        elif term == "XX":
            qc.h(target_qubits)
            qc_im.h(target_qubits)
        elif term == "YY":
            qc.sdg(target_qubits)
            qc.h(target_qubits)
            qc_im.sdg(target_qubits)
            qc_im.h(target_qubits)
        else:
            raise ValueError(f"Unsupported term type: {term}")
        if not(using_statevector):
            qc.measure_all()
            qc_im.measure_all()

        qc = qc.decompose(reps=5)
        qcs_re.append(qc)

        qc_im = qc_im.decompose(reps=5)
        qcs_im.append(qc_im)

    return None

def plot_convergence(ws, evals):
    cols = sns.color_palette("deep", n_colors=len(evals))
    fig = plt.figure(figsize=(8, 4))
    ax = fig.add_subplot(1, 1, 1)
    for it in range(len(ws)):
        x = it + 1
        for n in range(len(ws[it])):
            y = ws[it][n]
            ax.scatter(x, y, marker="D", color=cols[n])
    for n in range(len(evals)):
        ax.axhline(y=evals[n], color='gray', linestyle='dotted')
    ax.set_xlabel("iteration")
    ax.set_ylabel("overlap")
    ax.set_ylim(0, np.max(evals)*1.05)
    plt.show()
    # if using_statevector:
    #     fig.savefig("QKrylov_Norb"+str(Norb)+"Nocc"+str(Nocc)+"_statevector.pdf")
    # else:
    #     fig.savefig("QKrylov_Norb"+str(Norb)+"Nocc"+str(Nocc)+"_shot.pdf")
    plt.close()

def get_idx_circuit(op_string, term_types):
    idx_circuit = None
    if set(op_string) == {'I', 'Z'} or set(op_string) == {'I'}:                            
        for i in range(len(term_types)):
            if set(term_types[i]) == {'I', 'Z'} or set(term_types[i]) == {'I'}:
                idx_circuit = i
                break
        if idx_circuit is None:
            raise ValueError(f"Corresponding circuit for {op_string} not found in term_types: {term_types}")
    elif set(op_string) == {'X', 'I'} or set(op_string) == {'X'}:
        for i in range(len(term_types)):
            if set(term_types[i]) == {'X', 'I'} or set(term_types[i]) == {'X'}:
                idx_circuit = i
                break
        if idx_circuit is None:
            raise ValueError(f"Corresponding circuit for {op_string} not found in term_types: {term_types}")
    elif set(op_string) == {'Y', 'I'} or set(op_string) == {'Y'}:
        for i in range(len(term_types)):
            if set(term_types[i]) == {'Y', 'I'} or set(term_types[i]) == {'Y'}:
                idx_circuit = i
                break
        if idx_circuit is None:
            raise ValueError(f"Corresponding circuit for {op_string} not found in term_types: {term_types}")
    else:
        raise ValueError(f"Unexpected operator string: {op_string}. Supported types are 'IZ', 'XX', 'YY' for now.")
    return idx_circuit           



def QuantumKrylov(Uprep: QuantumCircuit, # circuit to prepare a reference state
                 hamiltonian_op: SparsePauliOp, # Hamiltonian operator
                 sampler,
                 delta_t=0.01,
                 max_iterations=10,
                 trotter_steps=1, 
                 trotter_order=1,
                 num_shot=10**4, 
                 ancilla_qubits=[], target_qubits=[],
                 using_statevector=False,
                 do_simulation=True,
                 ):
    Ntar = len(target_qubits)
    N = np.zeros((max_iterations, max_iterations), dtype=np.complex128)
    H = np.zeros((max_iterations, max_iterations), dtype=np.complex128)
    ws = []

    cPrep = make_cU(Uprep, None, Ntar, only_Uprep=True)

    # Assuming Toeplitz structure
    for it in tqdm(range(max_iterations)):
        print("iteration: ", it)

        ## evaluate overlap matrix element
        N[it, it] = 1.0

        # # Checking N is truelly identity at diagonal
        # if True:
        #     U_i = PauliEvolutionGate(hamiltonian_op, (it)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
        #     U_j = PauliEvolutionGate(hamiltonian_op, (it)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
        #     c_Ui = make_cU(Uprep, U_i, Ntar, only_Uprep=False)
        #     c_Uj = make_cU(Uprep, U_j, Ntar, only_Uprep=False)
        #     qc_re, qc_im = make_overlap_qc_conventional(Ntar, c_Ui, c_Uj, ancilla_qubits, target_qubits, using_statevector)
        #     results = [ Statevector.from_instruction(qc).probabilities_dict() for qc in [qc_re, qc_im]]
        #     prob_Re = results[0]
        #     prob_Im = results[1]
        #     p0 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Re.values()))
        #     p1 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Re.values()))
        #     ReN = p0 - p1
        #     p0 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Im.values()))
        #     p1 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Im.values()))
        #     ImN = p0 - p1
        #     print("Check N[", it, ",", it, "] ReN ", ReN, " ImN ", ImN)

        # ============== checking Toeplitz structure ==============
        if False :#True:
            U_i = PauliEvolutionGate(hamiltonian_op, (it)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
            U_j = PauliEvolutionGate(hamiltonian_op, (it+1)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
            c_Ui = make_cU(Uprep, U_i, Ntar, only_Uprep=False)
            c_Uj = make_cU(Uprep, U_j, Ntar, only_Uprep=False)
            qc_re, qc_im = make_overlap_qc_conventional(Ntar, c_Ui, c_Uj, ancilla_qubits, target_qubits, using_statevector)
            results = [ Statevector.from_instruction(qc).probabilities_dict() for qc in [qc_re, qc_im]]
            prob_Re = results[0]
            prob_Im = results[1]
            p0 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Re.values()))
            p1 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Re.values()))
            ReN = p0 - p1
            p0 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Im.values()))
            p1 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Im.values()))
            ImN = p0 - p1
            print("Check N_convention[", it, ",", it+1, "] ReN ", ReN, " ImN ", ImN)


        ## make controlled U_(k-j) = exp(-iHδt * (it+1))
        U_j = PauliEvolutionGate(hamiltonian_op, (it)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))

        for k in range(it):
            U_kj = PauliEvolutionGate(hamiltonian_op, (it-k)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
            U_j = PauliEvolutionGate(hamiltonian_op, (k)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
            qc_re, qc_im = make_overlap_qc(Ntar, cPrep, U_kj, U_j, ancilla_qubits, target_qubits, using_statevector)
            if using_statevector:
                results = [ Statevector.from_instruction(qc).probabilities_dict() for qc in [qc_re, qc_im]]
                prob_Re = results[0]
                prob_Im = results[1] 
            else:
                print("sampler used 2")
                job = sampler.run([qc_re, qc_im], shots=num_shot)
                results  = job.result()
                prob_Re = results[0].data.c.get_counts()
                prob_Im = results[1].data.c.get_counts()

            p0 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Re.values()))
            p1 = np.sum( [ count for bitstr, count in prob_Re.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Re.values()))
            ReN = p0 - p1

            p0 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '0' ] ) / np.sum(list(prob_Im.values()))
            p1 = np.sum( [ count for bitstr, count in prob_Im.items() if bitstr[-1] == '1' ] ) / np.sum(list(prob_Im.values()))
            ImN = p0 - p1

            N[k, it] = ReN + 1j * ImN
            N[it, k] = ReN - 1j * ImN
            print(f"N[{it}, {k}]", N[it, k])
            #if k == it-1:
            #    print("Check N_Toeplitz[", k, ",", it, "] ReN ", ReN, " ImN ", ImN)
                
        # print("N so far:")
        # for kk in range(it+1):
        #     print(N[kk, :it+1])
        ### evaluate H_ii no need ancilla qubit
        # ansatz, ansatz+All-H, ansatz+All-SdagH
        Ui = PauliEvolutionGate(hamiltonian_op, it*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
        qcs = [ ]
        term_types = { 0:"IZ", 1:"XX", 2:"YY"}  # general case => "XXYZ..."
        for idx_term in range(len(term_types)):
            term = term_types[idx_term]
            qc = QuantumCircuit(Ntar)
            qc.append(Uprep, range(Ntar))            
            qc.append(Ui, range(Ntar))
            if term == "IZ":
                pass
            elif term == "XX":
                qc.h(range(Ntar))
            elif term == "YY":
                qc.sdg(range(Ntar))
                qc.h(range(Ntar))
            else:
                raise ValueError(f"Unsupported term type: {term}")
            if not(using_statevector):
                qc.measure_all()
            qcs.append(qc)
                        
        if using_statevector:
            results = [ Statevector.from_instruction(qc).probabilities_dict() for qc in qcs]
        else:
            print("sampler used 3")
            job = sampler.run(qcs, shots=num_shot)
            results  = job.result()

        Hsum = 0.0
        for idx_H in range(len(Hamil_paulis)):
            op_string = Hamil_paulis[idx_H].to_label()
            idx_relevant = get_idx_to_measure(op_string)

            idx_circuit = get_idx_circuit(op_string, term_types)
            if using_statevector:
                res = results[idx_circuit]
            else:
                res = results[idx_circuit].data.meas.get_counts()
            expval, dummy, dummy_ = expec_Zstring(res, idx_relevant)
            Hsum += Hamil_coeffs[idx_H] * expval
            #print("operator: ", op_string, "coeff: ", Hamil_coeffs[idx_H], "exp. val: ",  expval)
        print(f"H[{it}, {it}]", Hsum)
        H[it, it] = Hsum

        if it > 0:
            ### evaluate H_ij ancilla qubit is needed
            for k in range(it): 
                qcs_re = []
                qcs_im = []
                U_kj = PauliEvolutionGate(hamiltonian_op, (it-k)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
                U_j = PauliEvolutionGate(hamiltonian_op, (k)*delta_t, synthesis=SuzukiTrotter(order=trotter_order,reps=trotter_steps))
                make_Circ_forNondiagH(term_types, Ntar, ancilla_qubits, target_qubits,
                                      cPrep, U_kj, U_j, qcs_re, qcs_im, using_statevector)

                # Re part
                if using_statevector:
                    results_Re = [ Statevector.from_instruction(qc).probabilities_dict() for qc in qcs_re]
                    results_Im = [ Statevector.from_instruction(qc).probabilities_dict() for qc in qcs_im]
                else:
                    print("sampler used 4")
                    job = sampler.run(qcs_re, shots=num_shot)
                    results_Re = job.result()
                    job = sampler.run(qcs_im, shots=num_shot)
                    results_Im = job.result()

                Re_H_ij = Im_H_ij = 0.0
                for idx_H in range(len(Hamil_paulis)):
                    op_string = Hamil_paulis[idx_H].to_label()
                    idx_circuit = get_idx_circuit(op_string, term_types)

                    idx_relevant = get_idx_to_measure(op_string)
                    if using_statevector:
                        res_Re = results_Re[idx_circuit]
                        res_Im = results_Im[idx_circuit]
                    else:
                        res_Re = results_Re[idx_circuit].data.meas.get_counts()
                        res_Im = results_Im[idx_circuit].data.meas.get_counts()
                    dummy_e, p0, p1 = expec_Zstring(res_Re, idx_relevant, target_qubits=range(len(target_qubits)), ancilla_qubit=0)
                    expval = p0 - p1
                    Re_H_ij += Hamil_coeffs[idx_H] * expval

                    dummy_e, p0, p1 = expec_Zstring(res_Im, idx_relevant, target_qubits=range(len(target_qubits)), ancilla_qubit=0)
                    expval = p0 - p1
                    Im_H_ij += Hamil_coeffs[idx_H] * expval
                H[k, it] = Re_H_ij + 1j * Im_H_ij
                H[it, k] = Re_H_ij - 1j * Im_H_ij
                print(f"H[{it}, {k}]", Re_H_ij - 1j * Im_H_ij)
        # for j in range(it-1, 0, -1):
        #     H[it, j] = H[it-1, j-1]
        #     H[j, it] = np.conj(H[it, j])
        #     print(f"H[{it},{j}] from H[{it-1},{j-1}] = {H[it, j]}")

        # print("H so far:")
        # for kk in range(it+1):
        #     print(H[kk, :it+1])

        # # # enforce Toeplitz structure
        # for j in range(1, it):
        #     H[it, j] = H[it-1, j-1]
        #     H[j, it] = np.conj(H[it, j])
        #     N[it, j] = N[it-1, j-1]
        #     N[j, it] = np.conj(N[it, j])

        #print("H after enforcing Toeplitz structure:")
        # for kk in range(it+1):
        #     print(H[kk, :it+1])

        # solve generalized eigenvalue problem
        Nsub = N[:it+1, :it+1]
        Hsub = H[:it+1, :it+1]        
        lam, v = scipy.linalg.eigh(Nsub) 
        # truncate orthogonal basis with small eigenvalues
        cols = [ i for i in range(it+1) if lam[i] >= 1.e-9]
        r = len(cols)
        Ur = v[:,cols]
        sq_Sigma_inv = np.diag(lam[cols]**(-0.5))

        X = Ur @ sq_Sigma_inv @ Ur.conj().T
        Xdag = X.conj().T
        tildeH = X @ Hsub @ Xdag
        w, v = scipy.linalg.eigh(tildeH)
        
        w_r = w[-r:]
        ws += [w_r]

        print("eigs of N: ", lam, "cond", np.linalg.cond(Nsub), "r:", r)
        print(f"w: {w_r}")
        print("")

    return H[:it+1,:it+1], N[:it+1,:it+1], ws

Trotter_order = 2
initial = "exact"
initial = ""

#using_statevector = False #True
using_statevector = True
sampler = Statevector if using_statevector else SamplerV2()

if initial == "exact":
    Uprep = ansatz(params_exact) # exact ground state
else:
    np.random.seed(0)
    params_random = np.random.rand(5)
    Uprep = ansatz(params_random) # random state

Hamil_coeffs = hamiltonian_op.coeffs
Hamil_paulis = hamiltonian_op.paulis
dt = 1.234 #0.345# 1.0


Hmat, Nmat, Ens = QuantumKrylov(Uprep, hamiltonian_op, sampler, dt, 
                                max_iterations=6,
                                trotter_order=Trotter_order,
                                trotter_steps=40, 
                                ancilla_qubits=[0], target_qubits=list(range(1,Norb+1)),
                                using_statevector=using_statevector)

print("Eexact", evals)
print("Nmat:")
print(Nmat)
print("Hmat:")
print(Hmat)

# Toeplitz & Hermitian check
for i in range(Hmat.shape[0]):
    for j in range(Hmat.shape[1]):
        if i >= 1 and j >=1 and i != j:
            diff_H = Hmat[i,j] - Hmat[i-1,j-1]
            diff_N = Nmat[i,j] - Nmat[i-1,j-1]
            if abs(diff_H) > 1e-6:
                print(f"Hmat not Toeplitz at ({i},{j}): diff {diff_H}")
            if abs(diff_N) > 1e-6:
                print(f"Nmat not Toeplitz at ({i},{j}): diff {diff_N}")
        if Hmat[i,j] != np.conj(Hmat[j,i]):
            print(f"Hmat not Hermitian at ({i},{j}): Hmat {Hmat[i,j]}, Hmat^* {np.conj(Hmat[j,i])}")
        if Nmat[i,j] != np.conj(Nmat[j,i]):
            print(f"Nmat not Hermitian at ({i},{j}): Nmat {Nmat[i,j]}, Nmat^* {np.conj(Nmat[j,i])}")


  0%|          | 0/6 [00:00<?, ?it/s]

iteration:  0
H[0, 0] (1.61127484567545+0j)
eigs of N:  [1.] cond 1.0 r: 1
w: [1.61127485]

iteration:  1
N[1, 0] (-0.02205787422924088+0.6840448486693917j)
H[1, 1] (1.6113755425167748+0j)


 33%|███▎      | 2/6 [00:05<00:11,  2.90s/it]

H[1, 0] (-0.2742973956846055+0.5443589163485792j)
eigs of N:  [0.3155996 1.6844004] cond 5.337143606763828 r: 2
w: [1.22663682 3.41188057]

iteration:  2
N[2, 0] (-0.8406458436837065+0.32452003180346833j)
N[2, 1] (-0.02205787422924088+0.6840448486693917j)
H[2, 2] (1.611712704125864+0j)
H[2, 0] (-1.0453061009597988+0.7045661378227288j)


 50%|█████     | 3/6 [00:17<00:19,  6.41s/it]

...

H[3, 2] (-0.2742973956846055+0.5443589163485792j)
eigs of N:  [0.0087001  0.02988648 0.63535308 3.32606034] cond 382.30150942803607 r: 4
w: [1.19275242 3.30543268 5.31634728 8.09686193]

iteration:  4
N[4, 0] (0.6188257511701873-0.4124592614948509j)
N[4, 1] (-0.09129948340044364-0.8343352082690203j)
N[4, 2] (-0.8406458436837065+0.32452003180346833j)
N[4, 3] (-0.02205787422924088+0.6840448486693917j)
H[4, 4] (1.6117622126715008+0j)
H[4, 0] (0.46415548786297517-0.7122906789335954j)
H[4, 1] (0.1902825478426009-1.0734411710944458j)
H[4, 2] (-1.0453061009597988+0.7045661378227288j)


 83%|████████▎ | 5/6 [00:56<00:14, 14.47s/it]

H[4, 3] (-0.2742973956846055+0.5443589163485792j)
eigs of N:  [1.68760117e-03 1.94524136e-02 3.68274453e-02 8.38597469e-01
 4.10343507e+00] cond 2431.519453840413 r: 5
w: [ 1.18974563  3.29858303  4.24157295  7.79777715 12.92074087]

iteration:  5
N[5, 0] (0.39456728144749226+0.9033814324633921j)
N[5, 1] (0.6188257511701873-0.4124592614948509j)
N[5, 2] (-0.09129948340044364-0.8343352082690203j)
N[5, 3] (-0.8406458436837065+0.32452003180346833j)
N[5, 4] (-0.02205787422924088+0.6840448486693917j)
H[5, 5] (1.6117325948397758+0j)
H[5, 0] (0.4806128383054936+1.4935442823877432j)
H[5, 1] (0.46415548786297517-0.7122906789335954j)
H[5, 2] (0.1902825478426009-1.0734411710944458j)
H[5, 3] (-1.0453061009597988+0.7045661378227288j)


100%|██████████| 6/6 [01:24<00:00, 14.02s/it]

H[5, 4] (-0.2742973956846055+0.5443589163485792j)
eigs of N:  [-7.49316977e-03  2.90603528e-03  2.29593103e-02  4.77518281e-02
  9.72664455e-01  4.96121154e+00] cond 1707.2096737608776 r: 5
w: [1.18976954 3.29739328 3.867977   7.39580036 9.0429682 ]

Eexact [1.18985184 3.29649666 5.34       5.34       7.42853393 9.44511758]
Nmat:
[[ 1.        +0.j         -0.02205787-0.68404485j -0.84064584-0.32452003j
  -0.09129948+0.83433521j  0.61882575+0.41245926j  0.39456728-0.90338143j]
 [-0.02205787+0.68404485j  1.        +0.j         -0.02205787-0.68404485j
  -0.84064584-0.32452003j -0.09129948+0.83433521j  0.61882575+0.41245926j]
 [-0.84064584+0.32452003j -0.02205787+0.68404485j  1.        +0.j
  -0.02205787-0.68404485j -0.84064584-0.32452003j -0.09129948+0.83433521j]
 [-0.09129948-0.83433521j -0.84064584+0.32452003j -0.02205787+0.68404485j
   1.        +0.j         -0.02205787-0.68404485j -0.84064584-0.32452003j]
 [ 0.61882575-0.41245926j -0.09129948-0.83433521j -0.84064584+0.32452003j
  -0.0